In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['proyecto_bigdata'] 
coleccion = db['datos_scrapingG3'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [8]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time #para capturar la hora y fecha de obtención

NOMBRE_GRUPO = "G3_FinanzasyMercado" #Agegaremos el nombre de grupo dentro del diccionario y la hora en que se obtubo

# --- PASO 1: CONFIGURACIÓN ---
options = Options()
options.add_argument("--headless") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

# --- PASO 2: INICIALIZACIÓN DE VARIABLES (LA MEMORIA) ---
datos_finales = []
limite_paginas = 3

try:
    driver.get("http://books.toscrape.com/")

    # Capturamos todas las categorías y guardamos sus nombres y URLs en una lista
    categorias = driver.find_elements(By.CSS_SELECTOR, "ul.nav-list ul li a")
    lista_categorias = [(cat.text.strip(), cat.get_attribute("href")) for cat in categorias]

    # Iteramos cada categoría automáticamente
    for nombre_categoria, url_categoria in lista_categorias:
        driver.get(url_categoria)
        print(f"Entrando a categoría: {nombre_categoria}")

        # Recorremos las páginas de cada categoría
        while True:
            WebDriverWait(driver, 10).until(
                EC.presence_of_all_elements_located((By.CLASS_NAME, "product_pod"))
            )
            bloques = driver.find_elements(By.CLASS_NAME, "product_pod")

            for bloque in bloques:
                titulo = bloque.find_element(By.TAG_NAME, "h3").find_element(By.TAG_NAME, "a").get_attribute("title")
                precio = bloque.find_element(By.CLASS_NAME, "price_color").text

                item_extraido = {
                    "item": titulo,
                    "valor": precio,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO,
                    "categoria": nombre_categoria
                }
                datos_finales.append(item_extraido)

            # Intentamos pasar a la siguiente página
            try:
                siguiente = driver.find_element(By.CSS_SELECTOR, "li.next a")
                siguiente.click()
            except:
                print(f"Fin de categoría {nombre_categoria}")
                break

    print(f"\nExtracción finalizada. Registros en memoria: {len(datos_finales)}")

except Exception as e:
    print(f"Error durante la extracción: {e}")


finally:
    driver.quit()

#TENEMOS EL DICCIONARIO CON UN FORMATO JSON, OBTENIDO DEL SCRIPT. AHORA LO INCLUIMOS EN LA BASE DE DATOS
#Cambios en los datos de salida
from pymongo import MongoClient



# 1. Configuracion de conexion
try:
    # 'database' es el nombre del servicio en el docker-compose
    client = MongoClient('database', 27017)
    db = client['proyecto_bigdata']
    coleccion = db['datos_scrapingG3']
    print("CONEXION ESTABLECIDA: Conectado a MongoDB exitosamente.")
except Exception as e:
    print("ERROR DE CONEXION:", e)

# 2. Procesamiento e insercion de datos
registros_exitosos = 0
registros_fallidos = 0

for dato in datos_finales:
    try:
        # Limpieza de simbolos y conversion a float
        # Se eliminan simbolos de moneda, comas y espacios en blanco
        valor_sucio = str(dato.get('valor', '0'))
        valor_limpio = valor_sucio.replace('£', '').replace(',', '').strip()
        
        dato['valor'] = float(valor_limpio)
        
        # Insercion individual
        coleccion.insert_one(dato)
        registros_exitosos += 1
        
    except ValueError:
        print("ADVERTENCIA: No se pudo procesar el valor:", valor_sucio)
        registros_fallidos += 1
    except Exception as e:
        print("ERROR EN INSERCION:", e)
        registros_fallidos += 1

print("RESUMEN DE CARGA:")
print("Registros exitosos:", registros_exitosos)
print("Registros fallidos:", registros_fallidos)

Entrando a categoría: Travel
Fin de categoría Travel
Entrando a categoría: Mystery
Fin de categoría Mystery
Entrando a categoría: Historical Fiction
Fin de categoría Historical Fiction
Entrando a categoría: Sequential Art
Fin de categoría Sequential Art
Entrando a categoría: Classics
Fin de categoría Classics
Entrando a categoría: Philosophy
Fin de categoría Philosophy
Entrando a categoría: Romance
Fin de categoría Romance
Entrando a categoría: Womens Fiction
Fin de categoría Womens Fiction
Entrando a categoría: Fiction
Fin de categoría Fiction
Entrando a categoría: Childrens
Fin de categoría Childrens
Entrando a categoría: Religion
Fin de categoría Religion
Entrando a categoría: Nonfiction
Fin de categoría Nonfiction
Entrando a categoría: Music
Fin de categoría Music
Entrando a categoría: Default
Fin de categoría Default
Entrando a categoría: Science Fiction
Fin de categoría Science Fiction
Entrando a categoría: Sports and Games
Fin de categoría Sports and Games
Entrando a categoría: 